<a href="https://colab.research.google.com/github/ghinanto/gnn_gw/blob/main/o4b_0_Triggers_and_Injections_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SETUP


In [43]:
%matplotlib inline
# if error run twice
! pip install -q pandas scipy matplotlib scikit-learn pycbc==2.8

In [44]:
# import libraries
import pycbc
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np

In [45]:
pd.set_option('display.float_format', '{:}'.format)#format to not have scientific notation. Add {:.4e} to have four decimals

# o4b-0 triggers from Sarah with configuration:


```
configuration = {
"window": 512,
"overlap":16, #here same region is covered 4 times, try lower
"threshold": 0.5,
"file": tmpffl,
"channel": ifo+"1:STRAIN_BURST_0",
"run":"full_print3",
"len":10.0,
"dir": '/srv/beegfs/dpnc/groups/gw/sarah/wdf_playground/',
"outdir": '/srv/beegfs/dpnc/groups/gw/sarah/wdf_playground/',
"ID":"WDF_test_"+ifo+"1_0",
"ARorder": 2000, # because real noise (fake noise 1k ok)
"learn": 200,
"preWhite":3,
"ResamplingFactor":8,
"LowFrequencyCut":12,
"FilterOrder":6,
"nproc":14,
'itf':ref,
"gps": min(after_veto)[0],
'lastGPS': max(after_veto)[1],
"segments":after_veto
}
```



Getting triggers from public repo (https://github.com/ghinanto/gnn_gw.git):

In [46]:
# get triggers from repo
triggers_files = {'H': "https://raw.githubusercontent.com/ghinanto/gnn_gw/refs/heads/main/triggers_hanford.csv", 'L': "https://raw.githubusercontent.com/ghinanto/gnn_gw/refs/heads/main/triggers_livingston.csv"} #, 'V': "triggers_virgo.csv"}
itf_sites_list = ['H', 'L'] #, 'V']
# read triggers in dataframe
triggers_df = {}
for ITF in itf_sites_list:
    triggers_df[ITF] = pd.read_csv(triggers_files[ITF])
#print(df_itf_data)

## How to access triggers

`triggers_df` is a dictionary containing a different trigger dataframe for each itf

In [47]:
triggers_df['H']

,gps,gpsPeak,duration,EnWDF,snrMean,snrPeak,freqMin,freqMean,freqMax,freqPeak,...,wt502,wt503,wt504,wt505,wt506,wt507,wt508,wt509,wt510,wt511
0,1402436049.859375,1402436050.0004883,0.0048828125,0.5207324367866512,0.4569136210357895,6.113423365875362,12.0,72.46153846153847,180.0,20.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1402436059.7890625,1402436060.0,0.0244140625,0.8333744468137334,0.8569261079001252,10.631658182974729,24.0,74.0,124.0,36.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1402436069.9609375,1402436070.0004883,0.01171875,1.4489854324471336,1.5769807067411394,18.53137287730165,8.0,69.53846153846153,120.0,36.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1402436089.8203125,1402436090.0004883,0.02685546875,0.8175907082013761,0.870263113806825,8.870364063960707,8.0,83.38461538461539,156.0,20.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1402436099.9921875,1402436100.0004883,0.01318359375,0.8634525556922379,1.0614104630719894,13.339421276071562,24.0,89.07692307692308,180.0,36.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14864,1402279653.7890625,1402279654.0205078,0.240234375,0.6376103790372831,0.3340220613253927,1.72929329681386,72.0,141.69230769230768,228.0,144.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14865,1402279656.9375,1402279657.0009766,0.017578125,1.5854428242505152,1.3041864451838794,15.241391028578208,28.0,78.15384615384616,132.0,60.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14866,1402279658.875,1402279658.8979492,0.24951171875,0.5454071035138536,0.3408257839064426,1.4236412301984829,52.0,119.6923076923077,232.0,124.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14867,1402279666.8671875,1402279667.0009766,0.01123046875,3.735620755795916,2.5900528470415085,30.42279644782136,4.0,55.23076923076923,132.0,68.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [48]:
triggers_df['L']

,gps,gpsPeak,duration,EnWDF,snrMean,snrPeak,freqMin,freqMean,freqMax,freqPeak,...,wt502,wt503,wt504,wt505,wt506,wt507,wt508,wt509,wt510,wt511
0,1402310586.0,1402310586.0004883,0.24951171875,1.299206437227144,1.119918671545818,12.482401159000672,4.0,58.76923076923077,132.0,40.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1402310625.9609375,1402310626.0009766,0.0048828125,1.6020211326571232,1.427294392842551,16.504489849454707,4.0,54.15384615384615,108.0,20.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1402310645.8203125,1402310646.0004883,0.0771484375,0.5787349799362784,0.4837856732435328,5.742278108360163,8.0,94.0,176.0,20.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1402310655.9921875,1402310656.0009766,0.2373046875,0.5999784829395222,0.5558256976162798,5.25676057950142,8.0,61.69230769230769,132.0,20.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1402310695.953125,1402310696.0009766,0.01220703125,1.2702218612961107,1.282895189124556,15.256674714224424,4.0,54.0,104.0,36.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12981,1402309893.8359375,1402309894.0,0.0068359375,0.5088894446724356,0.4064057419960049,5.046473347680055,136.0,186.0,236.0,168.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12982,1402336602.0,1402336602.0004883,0.24951171875,0.9254003473243436,0.8490432817498272,9.763185631435237,20.0,70.0,120.0,44.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12983,1402336631.7890625,1402336632.0004883,0.02490234375,1.019829156698124,1.0075594838688458,11.543435159544972,24.0,74.0,124.0,36.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12984,1402336651.890625,1402336652.0004883,0.00537109375,1.0524348968028765,1.115674552481518,12.944641992716017,12.0,62.0,112.0,20.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


We select triggers depending on parameter values using masks:

In [49]:
IFO='H'
mask_test = triggers_df[IFO]['snrPeak'] > 80
triggers_df[IFO][mask_test]

,gps,gpsPeak,duration,EnWDF,snrMean,snrPeak,freqMin,freqMean,freqMax,freqPeak,...,wt502,wt503,wt504,wt505,wt506,wt507,wt508,wt509,wt510,wt511
146,1402439029.0078125,1402439029.142578,0.0107421875,10.317918583912004,7.868437255328175,83.62065384498014,72.0,122.0,172.0,108.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
223,1402440824.1015625,1402440824.2617188,0.2314453125,4.431651750479376,11.303230819884284,138.74602577376874,24.0,116.0,292.0,40.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
224,1402440824.34375,1402440824.4750977,0.02392578125,39.99544999197884,132.14993214059774,1154.086275614853,32.0,82.3076923076923,140.0,48.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
292,1402442400.7421875,1402442400.9223633,0.03271484375,5.780441399017173,6.568384699591329,84.0731763218118,124.0,186.92307692307693,244.0,168.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
293,1402442400.984375,1402442401.1484375,0.07666015625,9.613889039378645,11.728973486833077,104.6677823197756,32.0,99.6923076923077,196.0,48.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12790,1402255996.3671875,1402255996.5307617,0.00390625,3.487422131315905,4.211156568125961,83.93223817859406,164.0,266.0,352.0,316.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12984,1402270962.890625,1402270963.0913086,0.0087890625,13.788513204576873,11.19270386441383,153.67551118171986,88.0,138.15384615384616,192.0,144.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14668,1402278977.359375,1402278977.5839844,0.20947265625,13.919136499856316,15.502302370969383,126.02783311741992,32.0,138.0,284.0,60.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14669,1402278977.6015625,1402278977.7827148,0.033203125,57.69906001120501,97.00403925802156,865.2708617923738,36.0,93.3846153846154,144.0,68.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# o4b-0 INJECTIONS

Download injections file from my dropbox (they are at CIT originally):

In [50]:
import requests

# Public sharing link
data_link = "https://www.dropbox.com/scl/fi/io7tz6z7w10p6c4l5vypb/burst_benchmark_short-0.h5?rlkey=xog18i7kls966dh4k3wh0f2z4&st=6fol2p3s&dl=1"
data_file = "burst_benchmark_short-0.h5"

# Download file
response = requests.get(data_link, allow_redirects=True, stream=True)
if response.status_code == 200:
    with open(data_file, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"File downloaded successfully as {data_file}")
else:
    print(f"Error: {response.status_code}, {response.text}")

File downloaded successfully as burst_benchmark_short-0.h5


Injection file refer to the entire o4b-2 (3months). Define start and end of **o4b-0** (3 days)

In [51]:
gpsStart_o4b0 = 1402200018
gpsEnd_o4b0 = 1402527618

The next cell read the `.h5` injection file and produces 3 objects (dictionaries) containing all the data in the form of DataFrames.

1.   `inj_parameters`: containing parameters of injections, e.g. time, ra, dec, etc. They vary for each injection type.
2.   `inj_attributes`: contining attributes of injections that is for the most only the injection type. If type is WAVE however, here you find also the `sampling_rate` and `hrss_norm` attributes.
3.   `inj_STRAIN_time_series`: only for type WAVE injections. Containing the time series defined by `hcross`, `hplus` and `time`. The size of this series (i.e. the number of samples) and the `sampling_rate` from `inj_attributes` gives the duration of the injection.

Let's run and talk about it with examples.

In [52]:
import h5py
import pandas as pd
import numpy as np

def inj_h5_to_df(inj_file):
  inj_parameters = {}
  inj_attributes = {}
  inj_STRAIN_time_series = {}
  #open file
  with h5py.File(inj_file, "r") as f:
    inj_wave_types = list(f.keys())

    # loop all inj types
    for inj_wave_type in inj_wave_types:
      # save parameters in dict
      inj_parameters[inj_wave_type] = pd.DataFrame(np.array(f[inj_wave_type]['PARAMETERS']))
      inj_attributes[inj_wave_type] = {inj_attr_key: inj_attr_val for inj_attr_key, inj_attr_val in zip(list(f[inj_wave_type].attrs), list([f[inj_wave_type].attrs[attr] for attr in f[inj_wave_type].attrs]))}

      if f[inj_wave_type].attrs['type'] == 'WAVE':
        # save other attrs
        # dataframe with columns of strain_pars
        inj_STRAIN_time_series[inj_wave_type] = pd.DataFrame()
        for strain_par in f[inj_wave_type]['STRAIN']:
          inj_STRAIN_time_series[inj_wave_type][strain_par] = np.array(f[inj_wave_type]['STRAIN'][strain_par]) # build columns of a dataframe with
    return inj_parameters, inj_attributes, inj_STRAIN_time_series

In [53]:
inj_parameters, inj_attributes, inj_STRAIN_time_series = inj_h5_to_df(data_file)

So each of them is a dictionary having injection wave types as keys, e.g. 'cbc_bbh10', 'ccsn_oco18_mesa20pertlr', 'sg_235q100'. Let's print `inj_parameters`:

In [54]:
inj_parameters

{'cbc_bbh10':                   time                 ra                   dec  \
 0   1401544616.6077347 0.6827237915976532 -0.002122447397676227   
 1   1401549018.8499951  3.293118558923391  0.018214685256745498   
 2    1401549361.914258 2.1744442699842184   0.13820620189023927   
 3   1401560966.6830468  6.033871590690589    -0.232330463206306   
 4   1401563965.2646575  3.173639928987707  -0.20820676987776812   
 ..                 ...                ...                   ...   
 312 1403872401.1521857  5.402287239269025  -0.36509205549652246   
 313  1403879908.391154  4.177788537859473    -1.176994094810024   
 314 1403887584.3378274  4.207592299758727     0.568964308094855   
 315 1403911011.6065469 1.4534266023218212    0.2283586802122033   
 316  1403963810.611586  2.024254197243319  -0.05115205979735421   
 
                    pol          amplitude   seed  
 0   2.4302941755495318  4.317994755509715 5533.0  
 1    5.380765731268311  1.783393898391357 5533.0  
 2   1.271443

As you can see, each injection type has different parameters, i.e. dataframes with different sizes/keys. The same for the other two:

In [55]:
inj_attributes

{'cbc_bbh10': {'hrss_norm': np.float64(1e-22),
  'sampling_rate': np.float64(16384.0),
  'type': 'WAVE'},
 'cbc_bbh125': {'hrss_norm': np.float64(1e-22),
  'sampling_rate': np.float64(16384.0),
  'type': 'WAVE'},
 'cbc_bbh35': {'hrss_norm': np.float64(1e-22),
  'sampling_rate': np.float64(16384.0),
  'type': 'WAVE'},
 'cbc_bbh50': {'hrss_norm': np.float64(1e-22),
  'sampling_rate': np.float64(16384.0),
  'type': 'WAVE'},
 'cbc_ebbh_m100q100e19': {'hrss_norm': np.float64(1e-22),
  'sampling_rate': np.float64(16384.0),
  'type': 'WAVE'},
 'cbc_ebbh_m100q50e88': {'hrss_norm': np.float64(1e-22),
  'sampling_rate': np.float64(16384.0),
  'type': 'WAVE'},
 'cbc_ebbh_m150q100e59': {'hrss_norm': np.float64(1e-22),
  'sampling_rate': np.float64(16384.0),
  'type': 'WAVE'},
 'cbc_ebbh_m175q50e51': {'hrss_norm': np.float64(1e-22),
  'sampling_rate': np.float64(16384.0),
  'type': 'WAVE'},
 'cbc_ebbh_m200q100e70': {'hrss_norm': np.float64(1e-22),
  'sampling_rate': np.float64(16384.0),
  'type': '

In [56]:
inj_STRAIN_time_series

{'cbc_bbh10':                        hcross                  hplus              times
 0      -1.139692032266351e-24 -1.536460582205865e-23  -22.0752505379375
 1      -1.198607722349395e-24 -1.536013235410152e-23 -22.07518950278125
 2      -1.257505919654757e-24  -1.53554329734781e-23   -22.075128467625
 3       -1.31638576599636e-24 -1.535050774360941e-23 -22.07506743246875
 4      -1.375246392458546e-24 -1.534535674030533e-23  -22.0750063973125
 ...                       ...                    ...                ...
 363798                    0.0                   -0.0 0.1292172355000005
 363799                    0.0                   -0.0 0.1292782706562505
 363800                    0.0                   -0.0 0.1293393058125005
 363801                    0.0                   -0.0 0.1294003409687505
 363802                    0.0                   -0.0 0.1294613761250005
 
 [363803 rows x 3 columns],
 'cbc_bbh125':                      hcross                  hplus             tim

Let's check on the keys

In [57]:
print("inj_parameters", inj_parameters.keys())
print(" each having different attributes, example ", list(inj_parameters['cbc_bbh10'].keys()))
print()
print("inj_attributes", inj_attributes.keys())
print(" each having different attributes, example ", list(inj_attributes['cbc_bbh10'].keys()))
print()
print("inj_STRAIN_time_series", inj_STRAIN_time_series.keys())
print(" each having ", list(inj_STRAIN_time_series['cbc_bbh10'].keys()))

inj_parameters dict_keys(['cbc_bbh10', 'cbc_bbh125', 'cbc_bbh35', 'cbc_bbh50', 'cbc_ebbh_m100q100e19', 'cbc_ebbh_m100q50e88', 'cbc_ebbh_m150q100e59', 'cbc_ebbh_m175q50e51', 'cbc_ebbh_m200q100e70', 'cbc_ebbh_m200q50e36', 'cbc_ebbh_m20q50e60', 'cbc_ebbh_m60q50e20', 'cbc_ebbh_m80q100e40', 'ccsn_and16_s20', 'ccsn_and16_s20s', 'ccsn_and19_s15fr', 'ccsn_mez23_d15', 'ccsn_mez23_d9', 'ccsn_mor18_m13', 'ccsn_oco18_mesa20pertlr', 'ccsn_pan18_s402d-dd2', 'ccsn_pan21_s40fr', 'ccsn_pow18_s18', 'ccsn_pow20_y20', 'ccsn_pow21_z100', 'ccsn_pow23_m39-1e12', 'ccsn_rad19_s10', 'che_1', 'che_2', 'che_3', 'cs_cusp', 'cs_kink', 'cs_kinkkink', 'ga_1', 'ga_100', 'ga_20', 'ga_3', 'proca_w85', 'sg_1304q9', 'sg_14q100', 'sg_1615q100', 'sg_196q700', 'sg_2000q3', 'sg_235q100', 'sg_235q3', 'sg_235q9', 'sg_2477q9', 'sg_25q9', 'sg_2634q700', 'sg_3067q3', 'sg_32q3', 'sg_40q700', 'sg_554q9', 'sg_612q700', 'sg_70q100', 'sg_70q3', 'sg_70q9', 'sg_849q3', 'wnb_100_120_t1000', 'wnb_100_200_t100', 'wnb_1220_1520_t500', 'wnb_1

Now let's keep only o4b-0 injections:

In [58]:
# keep only o4b-0 triggers
for inj_wave_type in inj_parameters: # loop over inj_parameters keys
  mask_o4b0 = (inj_parameters[inj_wave_type]["time"] > gpsStart_o4b0) & (inj_parameters[inj_wave_type]["time"] < gpsEnd_o4b0)
  inj_parameters[inj_wave_type] = inj_parameters[inj_wave_type][mask_o4b0]

## How to look for injection information:

Example. I want to get the injection with maximum amplitude of the type `ccsn_pan21_s40fr`:

In [59]:
inj_parameters['ccsn_pan21_s40fr'].loc[inj_parameters['ccsn_pan21_s40fr']['amplitude'].idxmax()]

,100
time,1402360714.3399901
ra,0.6646404911099573
dec,0.34313265791998454
pol,0.3148485673882525
amplitude,37.09340153372006
seed,5533.0


where we see the injection of maximum amplitude of the type `ccsn_pan21_s40fr` sits in index 100 of inj_parameters:

In [60]:
inj_parameters['ccsn_pan21_s40fr'].loc[100]

,100
time,1402360714.3399901
ra,0.6646404911099573
dec,0.34313265791998454
pol,0.3148485673882525
amplitude,37.09340153372006
seed,5533.0


This is a pd.Series, and values can be accessed simply by specifying the field:

In [61]:
inj_parameters['ccsn_pan21_s40fr'].loc[100]['time']

np.float64(1402360714.3399901)

Now let's say we want to combine injection parameters from different injection wave types in a single dataframe, for example to get for every inj wave type the injection with greater amplitude. There would be problems in that each injection type is described by different parameters. the best way I found is:

1.   fill a dictionary first, where the keys will be the row indices of the DataFrame, and the content a pd.Series that will become the actual row itself. The column indices will be merged, filling with NaN where some of the rows don't have that particular field.
2.   just input the dictionary to the DataFrame constructor

So we get:

In [62]:
inj_parameters_max_amp_dict = {}
# get injection with maximum amplitude for each wave type
for inj_wave_type in inj_parameters:
  inj_parameters_max_amp_dict[inj_wave_type] = inj_parameters[inj_wave_type].loc[inj_parameters[inj_wave_type]['amplitude'].idxmax()]

inj_parameters_max_amp_df = pd.DataFrame(inj_parameters_max_amp_dict).transpose().sort_values(by='amplitude', ascending=False)
del inj_parameters_max_amp_dict
inj_parameters_max_amp_df

,amplitude,bandwidth,dec,duration,eccentricity,frequency,phase,pol,quality,ra,seed,time
cbc_bbh125,164.44454677050774,NaN,0.0840205511813443,NaN,NaN,NaN,NaN,3.2840321996684776,NaN,6.253496171322681,5533.0,1402327559.321994
ccsn_pan18_s402d-dd2,46.80541788218131,NaN,1.1350370134192302,NaN,NaN,NaN,NaN,1.3457838921138343,NaN,4.985074553850091,5533.0,1402265088.2911243
ccsn_pan21_s40fr,37.09340153372006,NaN,0.34313265791998454,NaN,NaN,NaN,NaN,0.3148485673882525,NaN,0.6646404911099573,5533.0,1402360714.3399901
che_2,16.11433705845197,NaN,1.339366338509362,NaN,NaN,NaN,NaN,5.626050258713189,NaN,3.5142461185502762,5533.0,1402223251.5243294
cbc_bbh10,12.341295080615499,NaN,-1.0348338801848587,NaN,NaN,NaN,NaN,3.545925523525306,NaN,6.1649265908343995,5533.0,1402315715.86411
...,...,...,...,...,...,...,...,...,...,...,...,...
wnb_295_305_t400,6.562028530961173e-38,10.0,-0.734666220887999,0.4,0.5880792951065662,300.0,3.1998512345981465,5.319442942774546,NaN,2.667109595661014,5533.0,1402466246.773795
wnb_80_90_t200,3.4178839488728417e-38,10.0,-1.0404112260739846,0.2,0.1303236925573562,85.0,0.0981650844632566,1.2392608511518273,NaN,5.100703436932842,5533.0,1402333924.2166963
wnb_100_200_t100,2.5904886308517063e-38,100.0,-0.10230667527534433,0.1,0.10600013736342928,150.0,4.963929238901601,1.913116401414095,NaN,5.54841404160853,5533.0,1402244030.5541804
wnb_30_50_t1800,2.1357822601759149e-38,20.0,0.016727086786663764,1.8,0.13075236725389372,40.0,3.5101143809839783,5.509729523949598,NaN,5.244397164720279,5533.0,1402308080.1382647


So to get some parameter:

In [63]:
inj_parameters_max_amp_df.loc['cbc_bbh125']['time']

np.float64(1402327559.321994)

Now let's check if this inj type `cbc_bbh125` is of type WAVE, i.e. has a duration described by a time series. Let's print its `inj_attributes`:

In [64]:
inj_attributes['cbc_bbh125']

{'hrss_norm': np.float64(1e-22),
 'sampling_rate': np.float64(16384.0),
 'type': 'WAVE'}

In [65]:
if inj_attributes['cbc_bbh125']['type'] == 'WAVE':
  print("Injection 'cbc_bbh125' described by a time series, check inj_STRAIN_time_series!")
  display(inj_STRAIN_time_series['cbc_bbh125'])


Injection 'cbc_bbh125' described by a time series, check inj_STRAIN_time_series!


,hcross,hplus,times
0,2.415735850416778e-22,1.346098112206854e-22,-0.0460205078125
1,2.419119100312408e-22,1.340086374837242e-22,-0.04595947265625
2,2.422487840161079e-22,1.334064681801457e-22,-0.0458984375
3,2.425842037533694e-22,1.328033066460533e-22,-0.04583740234375
4,2.429181660069641e-22,1.321991562284579e-22,-0.0457763671875
...,...,...,...
33945,1.154926687588049e-26,-4.073621858212475e-27,2.02581787109375
33946,1.154927996631586e-26,-4.073519443521208e-27,2.02587890625
33947,1.154929295766943e-26,-4.073417053059075e-27,2.02593994140625
33948,1.154930587793702e-26,-4.073314679752676e-27,2.0260009765625


So what's the duration of the injection of maximum amplitude of type `cbc_bbh125`?

In [66]:
def inj_duration(inj_wave_type):
  if inj_attributes[inj_wave_type]['type'] != 'WAVE':
    print("This injection type has no waveform described by a time series!")
    return None
  return(len(inj_STRAIN_time_series[inj_wave_type]) / inj_attributes[inj_wave_type]['sampling_rate'])

In [67]:
# duration of the time series
print("Injection duration is:", inj_duration('cbc_bbh125'), "s")

Injection duration is: 2.0721435546875 s


## INJ TIME at the ITFs

The injection times are defined at the centre of the earth. So depending on the orientation and propagation direction of the grav wave the actual time at the interferometers can be greater of lesser.

This function returns a dictionary with an entry for each itf, with the corresponding time.

In [68]:
from pycbc.detector import Detector
def inj2ifo_time(inj_ra, inj_dec, inj_time_gps, ifo_list=['H1', 'L1']):
  ifo_inj_time = {}

  for IFO in ifo_list:
    det = Detector(IFO)
    #print("inj_max_amp_time_delay_from", IFO, ":", det.time_delay_from_earth_center(inj_ra, inj_dec, inj_time_gps), "s")
    ifo_inj_time[IFO] = inj_time_gps + det.time_delay_from_earth_center(inj_ra, inj_dec, inj_time_gps)

  return ifo_inj_time

For example, let's add to the previous dataframe tthe fields for the time of the innjections at the itfs:

In [69]:
inj_parameters_max_amp_df['time_at_H1'] = None
inj_parameters_max_amp_df['time_at_L1'] = None

for inj_wave_type, inj_parameters_max_amp in inj_parameters_max_amp_df.iterrows():
  inj_max_amp_time_at_IFO = inj2ifo_time(inj_parameters_max_amp_df.loc[inj_wave_type]['ra'], inj_parameters_max_amp_df.loc[inj_wave_type]['dec'], inj_parameters_max_amp_df.loc[inj_wave_type]['time'])
  for IFO in inj_max_amp_time_at_IFO:
    inj_parameters_max_amp_df.loc[inj_wave_type, 'time_at_'+IFO] = inj_max_amp_time_at_IFO[IFO]

inj_parameters_max_amp_df

,amplitude,bandwidth,dec,duration,eccentricity,frequency,phase,pol,quality,ra,seed,time,time_at_H1,time_at_L1
cbc_bbh125,164.44454677050774,NaN,0.0840205511813443,NaN,NaN,NaN,NaN,3.2840321996684776,NaN,6.253496171322681,5533.0,1402327559.321994,1402327559.3066525,1402327559.3081243
ccsn_pan18_s402d-dd2,46.80541788218131,NaN,1.1350370134192302,NaN,NaN,NaN,NaN,1.3457838921138343,NaN,4.985074553850091,5533.0,1402265088.2911243,1402265088.2833533,1402265088.2875996
ccsn_pan21_s40fr,37.09340153372006,NaN,0.34313265791998454,NaN,NaN,NaN,NaN,0.3148485673882525,NaN,0.6646404911099573,5533.0,1402360714.3399901,1402360714.3406482,1402360714.3502567
che_2,16.11433705845197,NaN,1.339366338509362,NaN,NaN,NaN,NaN,5.626050258713189,NaN,3.5142461185502762,5533.0,1402223251.5243294,1402223251.5098212,1402223251.5163288
cbc_bbh10,12.341295080615499,NaN,-1.0348338801848587,NaN,NaN,NaN,NaN,3.545925523525306,NaN,6.1649265908343995,5533.0,1402315715.86411,1402315715.8706987,1402315715.863994
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
wnb_295_305_t400,6.562028530961173e-38,10.0,-0.734666220887999,0.4,0.5880792951065662,300.0,3.1998512345981465,5.319442942774546,NaN,2.667109595661014,5533.0,1402466246.773795,1402466246.7823927,1402466246.785595
wnb_80_90_t200,3.4178839488728417e-38,10.0,-1.0404112260739846,0.2,0.1303236925573562,85.0,0.0981650844632566,1.2392608511518273,NaN,5.100703436932842,5533.0,1402333924.2166963,1402333924.2323265,1402333924.2328043
wnb_100_200_t100,2.5904886308517063e-38,100.0,-0.10230667527534433,0.1,0.10600013736342928,150.0,4.963929238901601,1.913116401414095,NaN,5.54841404160853,5533.0,1402244030.5541804,1402244030.5501726,1402244030.5572472
wnb_30_50_t1800,2.1357822601759149e-38,20.0,0.016727086786663764,1.8,0.13075236725389372,40.0,3.5101143809839783,5.509729523949598,NaN,5.244397164720279,5533.0,1402308080.1382647,1402308080.1234498,1402308080.1209948


## Let's search triggers around injections

Let's search for each of the injections of the `inj_parameters_max_amp_df` df

In [70]:
time_window_around_inj = 5#s
for inj_wave_type, inj_parameters_max_amp in inj_parameters_max_amp_df.iterrows():
  print("--------------------------->INJECTION TYPE: ", inj_wave_type)
  print("maximum amplitude INJECTION parameters:")
  display(inj_parameters_max_amp_df.loc[inj_wave_type].dropna().to_frame().transpose())
  print()
  for IFO in ['H', 'L']:
    mask_triggers_near_inj = (triggers_df[IFO]['gpsPeak'] < inj_parameters_max_amp['time_at_'+IFO+'1'] + time_window_around_inj) & (triggers_df[IFO]['gpsPeak'] > inj_parameters_max_amp['time_at_'+IFO+'1'] - time_window_around_inj)
    print("IFO:", IFO,', TRIGGERS:', len(triggers_df[IFO][mask_triggers_near_inj]), "found in a time window of ", time_window_around_inj, "s around the maximum amplitude injection of type", inj_wave_type)

    for trigger_index, trigger_parameters in triggers_df[IFO][mask_triggers_near_inj].iterrows():
      print("trigger number:", trigger_index, ", delay from inj:", abs(trigger_parameters['gpsPeak']-inj_parameters_max_amp['time_at_'+IFO+'1']), 's')
    print("TRIGGER parameters:")
    display(triggers_df[IFO][mask_triggers_near_inj][triggers_df[IFO].columns[:triggers_df[IFO].columns.get_loc('wt0')]])
    print()
  print()

--------------------------->INJECTION TYPE:  cbc_bbh125
maximum amplitude INJECTION parameters:


,amplitude,dec,pol,ra,seed,time,time_at_H1,time_at_L1
cbc_bbh125,164.44454677050774,0.0840205511813443,3.2840321996684776,6.253496171322681,5533.0,1402327559.321994,1402327559.3066525,1402327559.3081243



IFO: H , TRIGGERS: 0 found in a time window of  5 s around the maximum amplitude injection of type cbc_bbh125
TRIGGER parameters:


,gps,gpsPeak,duration,EnWDF,snrMean,snrPeak,freqMin,freqMean,freqMax,freqPeak,wave



IFO: L , TRIGGERS: 1 found in a time window of  5 s around the maximum amplitude injection of type cbc_bbh125
trigger number: 1327 , delay from inj: 3.306659460067749 s
TRIGGER parameters:


...

,amplitude,bandwidth,dec,duration,eccentricity,frequency,phase,pol,ra,seed,time,time_at_H1,time_at_L1
wnb_55_65_t500,7.27556104868897e-39,10.0,-0.08917874299731875,0.5,0.17038451961513013,60.0,0.09727857792008032,5.978927664287567,1.08750854577805,5533.0,1402319545.1941736,1402319545.1933384,1402319545.1840665



IFO: H , TRIGGERS: 0 found in a time window of  5 s around the maximum amplitude injection of type wnb_55_65_t500
TRIGGER parameters:


,gps,gpsPeak,duration,EnWDF,snrMean,snrPeak,freqMin,freqMean,freqMax,freqPeak,wave



IFO: L , TRIGGERS: 0 found in a time window of  5 s around the maximum amplitude injection of type wnb_55_65_t500
TRIGGER parameters:


,gps,gpsPeak,duration,EnWDF,snrMean,snrPeak,freqMin,freqMean,freqMax,freqPeak,wave


# Selected injections:


All with high SNR,
*   Very short injection (ga_1, ..., ga_100):
`inj_parameters['ga_100'].loc[98]`
*   ccsn:
`inj_parameters['ga_100'].loc[98]`
*   cbc_bbh125:
`inj_parameters['cbc_bbh125'].loc[105]`




Let's select the third highest amplitude inection of type `ga_100`:

In [71]:
inj_parameters['ga_100'].sort_values(by='amplitude', ascending=False).iloc[2]
#also by
#inj_parameters['ga_100'].loc[98]

,98
time,1402228309.881055
ra,1.056891437519439
dec,0.9993716959996298
pol,4.348741504816312
amplitude,1.6970549821006593e-16
seed,5533.0
duration,0.1


In [72]:
#so get time at the itfs as
time_ga100_98_atIFO = inj2ifo_time(inj_parameters['ga_100'].loc[98]['ra'], inj_parameters['ga_100'].loc[98]['dec'], inj_parameters['ga_100'].loc[98]['time'])
print("inj_parameters['ga_100'].loc[98]")
print("Time at H:", time_ga100_98_atIFO['H1'])
print("Time at L:", time_ga100_98_atIFO['L1'])

inj_parameters['ga_100'].loc[98]
Time at H: 1402228309.8696325
Time at L: 1402228309.8689678


In [73]:
inj_duration('ga_100')
print("duration:", inj_parameters['ga_100'].loc[98]['duration'], 's')

This injection type has no waveform described by a time series!
duration: 0.1 s


Let's take the highest amplitude injection from `ccsn_pan18_s402d-dd2`:

In [74]:
inj_parameters['ccsn_pan18_s402d-dd2'].sort_values(by='amplitude', ascending=False).iloc[0]
#also inj_parameters['ccsn_pan18_s402d-dd2'].loc[92]


,92
time,1402265088.2911243
ra,4.985074553850091
dec,1.1350370134192302
pol,1.3457838921138343
amplitude,46.80541788218131
seed,5533.0


In [75]:
#so get time at the itfs as
time_ccsn_pan18_s402d_dd2_92_atIFO = inj2ifo_time(inj_parameters['ccsn_pan18_s402d-dd2'].loc[92]['ra'], inj_parameters['ccsn_pan18_s402d-dd2'].loc[92]['dec'], inj_parameters['ccsn_pan18_s402d-dd2'].loc[92]['time'])
print("inj_parameters['ccsn_pan18_s402d-dd2'].loc[92]")
print("Time at H:", time_ccsn_pan18_s402d_dd2_92_atIFO['H1'])
print("Time at L:", time_ccsn_pan18_s402d_dd2_92_atIFO['L1'])

inj_parameters['ccsn_pan18_s402d-dd2'].loc[92]
Time at H: 1402265088.2833533
Time at L: 1402265088.2875996


In [76]:
print("inj duration from time series for type ccsn_pan18_s402d-dd2:")
print(inj_duration('ccsn_pan18_s402d-dd2'), 's')

inj duration from time series for type ccsn_pan18_s402d-dd2:
94.77391148001057 s


Let's take highest amplitude inj of type `cbc_bbh125`:

In [77]:
inj_parameters['cbc_bbh125'].sort_values(by='amplitude', ascending=False).iloc[0]
#also inj_parameters['cbc_bbh125'].loc[105]

,105
time,1402327559.321994
ra,6.253496171322681
dec,0.0840205511813443
pol,3.2840321996684776
amplitude,164.44454677050774
seed,5533.0


In [78]:
#so get time at the itfs as
time_cbc_bbh125_105_atIFO = inj2ifo_time(inj_parameters['cbc_bbh125'].loc[105]['ra'], inj_parameters['cbc_bbh125'].loc[105]['dec'], inj_parameters['cbc_bbh125'].loc[105]['time'])
print("inj_parameters['cbc_bbh125'].loc[105]")
print("Time at H:", time_cbc_bbh125_105_atIFO['H1'])
print("Time at L:", time_cbc_bbh125_105_atIFO['L1'])

inj_parameters['cbc_bbh125'].loc[105]
Time at H: 1402327559.3066525
Time at L: 1402327559.3081243


In [79]:
print("inj duration from time series for type cbc_bbh125:")
print(inj_duration('cbc_bbh125'), 's')

inj duration from time series for type cbc_bbh125:
2.0721435546875 s


### selecting ccsn inj with high snr

In [80]:
[wave_type for wave_type in inj_parameters_max_amp_df.index if 'ccsn' in wave_type]

['ccsn_pan18_s402d-dd2',
 'ccsn_pan21_s40fr',
 'ccsn_rad19_s10',
 'ccsn_and16_s20s',
 'ccsn_mez23_d15',
 'ccsn_pow21_z100',
 'ccsn_pow18_s18',
 'ccsn_and16_s20',
 'ccsn_oco18_mesa20pertlr',
 'ccsn_pow20_y20',
 'ccsn_mez23_d9',
 'ccsn_mor18_m13',
 'ccsn_pow23_m39-1e12',
 'ccsn_and19_s15fr']

In [81]:
ccsn_wave_types = [wave_type for wave_type in inj_parameters_max_amp_df.index if 'ccsn' in wave_type]
mask_ccsn = inj_parameters_max_amp_df.index
inj_parameters_max_amp_df.loc[ccsn_wave_types].sort_values(by='amplitude', ascending=False)

,amplitude,bandwidth,dec,duration,eccentricity,frequency,phase,pol,quality,ra,seed,time,time_at_H1,time_at_L1
ccsn_pan18_s402d-dd2,46.80541788218131,NaN,1.1350370134192302,NaN,NaN,NaN,NaN,1.3457838921138343,NaN,4.985074553850091,5533.0,1402265088.2911243,1402265088.2833533,1402265088.2875996
ccsn_pan21_s40fr,37.09340153372006,NaN,0.34313265791998454,NaN,NaN,NaN,NaN,0.3148485673882525,NaN,0.6646404911099573,5533.0,1402360714.3399901,1402360714.3406482,1402360714.3502567
ccsn_rad19_s10,9.626321196865183,NaN,-0.32136562421368314,NaN,NaN,NaN,NaN,2.0916972796428803,NaN,5.443197490092411,5533.0,1402283886.2370512,1402283886.2488406,1402283886.2408297
ccsn_and16_s20s,4.963481917641489,NaN,-0.5325391973932887,NaN,NaN,NaN,NaN,1.3113509163180495,NaN,5.833371369635343,5533.0,1402414826.3499827,1402414826.348803,1402414826.3509479
ccsn_mez23_d15,4.484278038815722,NaN,0.11088941735309783,NaN,NaN,NaN,NaN,4.007546467344519,NaN,5.085271592914525,5533.0,1402510095.6324604,1402510095.6392856,1402510095.6477
ccsn_pow21_z100,4.2718540685911375,NaN,0.520115679092806,NaN,NaN,NaN,NaN,1.6261394041355735,NaN,6.2752511925994705,5533.0,1402329468.2498376,1402329468.2304726,1402329468.2345915
ccsn_pow18_s18,3.7275758043187364,NaN,-0.5221963659300303,NaN,NaN,NaN,NaN,0.6992489779439639,NaN,4.531164377974312,5533.0,1402224311.1221492,1402224311.1205652,1402224311.1226268
ccsn_and16_s20,3.640478633532416,NaN,-0.2725506218160261,NaN,NaN,NaN,NaN,2.563629787141858,NaN,5.339998048930936,5533.0,1402270787.0115442,1402270787.0295422,1402270787.027981
ccsn_oco18_mesa20pertlr,3.293037142548756,NaN,0.07492665659407372,NaN,NaN,NaN,NaN,0.8621922046295688,NaN,1.4114300046850987,5533.0,1402404381.6006348,1402404381.6033862,1402404381.595659
ccsn_pow20_y20,3.1768657566183793,NaN,1.116862430843186,NaN,NaN,NaN,NaN,2.844234149557446,NaN,4.159996087381149,5533.0,1402487668.1879172,1402487668.1735704,1402487668.181481


In [82]:
inj_parameters['ccsn_pan18_s402d-dd2'].sort_values(by='amplitude', ascending=False)

,time,ra,dec,pol,amplitude,seed
92,1402265088.2911243,4.985074553850091,1.1350370134192302,1.3457838921138343,46.80541788218131,5533.0
114,1402402997.256765,1.7108314916069742,-1.000445343364793,4.1551026020587045,36.43205911084158,5533.0
98,1402323645.3167887,4.326875592545047,0.6565716288787146,0.8434583973517206,34.186328423144246,5533.0
127,1402520011.1484146,3.5999988031413444,-0.444921592123169,1.6721805622465808,29.975838704342685,5533.0
93,1402265301.5871384,2.2784092570409498,0.4877673539767966,3.8416520557153033,29.81204177832905,5533.0
87,1402233674.4077346,5.6079942795206,-0.9164697941081195,3.353275389927886,28.819771704756054,5533.0
124,1402503900.2693117,4.972453695445792,1.2046585976756592,0.9990423152911515,28.549105206537348,5533.0
99,1402324624.1353953,6.189353812878404,-0.13523171311431678,5.334010263828127,25.96744949158133,5533.0
101,1402342792.9154813,3.76383113109231,-0.49446368476374997,3.22373738943216,25.926995209264746,5533.0
108,1402379067.1589377,3.217499211344634,-0.5959067961854039,4.081608113969432,21.89570475706463,5533.0


### selecting short inj with high ampltiude

In [83]:
mask_short = inj_parameters_max_amp_df['duration'] < 1.
inj_parameters_max_amp_df[mask_short].sort_values(by='amplitude', ascending=False)

,amplitude,bandwidth,dec,duration,eccentricity,frequency,phase,pol,quality,ra,seed,time,time_at_H1,time_at_L1
ga_100,1.994540253969544e-16,NaN,-1.1514165528708418,0.1,NaN,NaN,NaN,3.6957906163861614,NaN,0.7996230162917786,5533.0,1402397283.7754822,1402397283.7905407,1402397283.7829256
ga_20,2.715934686207109e-20,NaN,1.1738824812375608,0.02,NaN,NaN,NaN,4.792634243939587,NaN,0.17177319496836016,5533.0,1402335239.2306128,1402335239.2119095,1402335239.2177355
ga_3,2.887139490407032e-22,NaN,0.7982272915261704,0.003,NaN,NaN,NaN,1.6183439788231966,NaN,0.26171958426613195,5533.0,1402485774.624565,1402485774.6083755,1402485774.6058788
ga_1,1.4710680826071645e-22,NaN,-1.3304068171871026,0.001,NaN,NaN,NaN,5.6252797285300735,NaN,3.2877168597827255,5533.0,1402360912.5626009,1402360912.5746262,1402360912.5687046
wnb_1220_1520_t500,6.378558632907808e-35,300.0,1.3918171972943447,0.5,0.6415628276200178,1370.0,5.5602434267526,3.494904360274674,NaN,1.003318495867917,5533.0,1402239805.5893328,1402239805.5725412,1402239805.5757031
wnb_1395_1405_t100,6.879192123620501e-36,10.0,-0.6255313994495769,0.1,0.9645047791821774,1400.0,2.3455110248553486,1.2809551080770805,NaN,4.616725682691891,5533.0,1402507605.832664,1402507605.8510518,1402507605.8536344
wnb_800_1000_t10,1.1094602486912695e-36,200.0,1.0378489591378561,0.01,0.4957653494097891,900.0,4.182799378412639,4.444968469244686,NaN,1.0589362172703265,5533.0,1402350956.880693,1402350956.8628356,1402350956.8698392
wnb_750_850_t100,1.0766345292885812e-36,100.0,-0.6648383711603701,0.1,0.11226566654657422,800.0,1.5466341401942587,6.061323066391147,NaN,4.686756021847648,5533.0,1402284521.8444858,1402284521.8507466,1402284521.8409636
wnb_250_350_t800,4.543039472454408e-37,100.0,-0.4908328075145552,0.8,0.1115327357938406,300.0,5.995668711207308,2.3967867905339717,NaN,5.188489318282579,5533.0,1402474443.4133298,1402474443.4092517,1402474443.4022293
wnb_400_450_t300,2.9562247963907372e-37,50.0,0.06748415830304533,0.3,0.6422955952771118,425.0,5.423118968065654,5.597107951670528,NaN,2.197470467302672,5533.0,1402294677.4789786,1402294677.483233,1402294677.4922254


In [84]:
inj_parameters['ga_100'].sort_values(by='amplitude', ascending=False)

,time,ra,dec,pol,amplitude,seed,duration
116,1402397283.7754822,0.7996230162917786,-1.1514165528708418,3.6957906163861614,1.994540253969544e-16,5533.0,0.1
110,1402332658.2450702,6.24573783833062,0.8330253029547856,2.7678017665554755,1.8810738341981743e-16,5533.0,0.1
98,1402228309.881055,1.056891437519439,0.9993716959996298,4.348741504816312,1.6970549821006593e-16,5533.0,0.1
119,1402409770.2171938,0.9360731156950961,0.8616266640204951,0.10171000791095232,1.605324018293499e-16,5533.0,0.1
102,1402249173.7027583,4.5777476002447735,0.30709194163246495,0.10567978413284584,1.4832365020747966e-16,5533.0,0.1
133,1402510988.3086734,0.44038639147287834,0.584614709036814,2.9345972892601324,1.4165814102161208e-16,5533.0,0.1
122,1402454198.969986,0.216471766590353,0.32769996279220515,4.060256757219871,1.3911708165660917e-16,5533.0,0.1
107,1402316765.2607918,4.488266280294372,-0.776511340997282,5.315830542388185,1.3417238038756367e-16,5533.0,0.1
128,1402491326.7502427,1.5339558122569292,-0.12581805832479054,2.4764557443317874,1.2842115698305946e-16,5533.0,0.1
101,1402247139.5077384,3.605733727193006,-0.3496437120327271,2.1947464978560194,1.2365332295119414e-16,5533.0,0.1
